In [1]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "CLAUDE.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

df = pd.read_parquet(
    PROJECT_ROOT / "data" / "raw" / "statcast_2024-04-15_ingested_2026-08-31.parquet"
).sort_values(["game_pk", "at_bat_number", "pitch_number"]).reset_index(drop=True)

print(df["pitch_type"].value_counts(dropna=False))
print()
print("missing:", df["pitch_type"].isna().sum())
print()
print(df["pitch_name"].value_counts(dropna=False) if "pitch_name" in df.columns else "no pitch_name column")

pitch_type
FF     1346
SI      698
SL      596
CH      466
FC      414
CU      314
ST      232
KC      136
FS      119
SV       25
NaN      16
Name: count, dtype: int64

missing: 16

pitch_name
4-Seam Fastball    1346
Sinker              698
Slider              596
Changeup            466
Cutter              414
Curveball           314
Sweeper             232
Knuckle Curve       136
Split-Finger        119
Slurve               25
NaN                  16
Name: count, dtype: int64


In [2]:
if "pitch_name" in df.columns:
    mapping = (
        df.groupby("pitch_type")["pitch_name"]
        .agg(["nunique", "first", "count"])
        .sort_values("count", ascending=False)
    )
    print(mapping.to_string())

            nunique            first  count
pitch_type                                 
FF                1  4-Seam Fastball   1346
SI                1           Sinker    698
SL                1           Slider    596
CH                1         Changeup    466
FC                1           Cutter    414
CU                1        Curveball    314
ST                1          Sweeper    232
KC                1    Knuckle Curve    136
FS                1     Split-Finger    119
SV                1           Slurve     25


In [3]:
SWING = {"foul", "hit_into_play", "swinging_strike",
         "swinging_strike_blocked", "foul_tip"}
WHIFF = {"swinging_strike", "swinging_strike_blocked"}

df["is_swing"] = df["description"].isin(SWING)
df["is_whiff"] = df["description"].isin(WHIFF)

by_pitch = (
    df.groupby("pitch_type", dropna=False)
    .agg(
        pitches=("is_swing", "size"),
        swings=("is_swing", "sum"),
        whiffs=("is_whiff", "sum"),
        avg_velo=("release_speed", "mean"),
    )
)
by_pitch["usage_pct"] = by_pitch["pitches"] / len(df)
by_pitch["swing_pct"] = by_pitch["swings"] / by_pitch["pitches"]
by_pitch["whiff_pct"] = by_pitch["whiffs"] / by_pitch["swings"]

print(by_pitch.sort_values("pitches", ascending=False).round(3).to_string())

            pitches  swings  whiffs  avg_velo  usage_pct  swing_pct  whiff_pct
pitch_type                                                                    
FF             1346     626     103    94.218      0.309      0.465      0.165
SI              698     314      34      92.9      0.160      0.450      0.108
SL              596     307      99    85.732      0.137      0.515      0.322
CH              466     236      74    86.198      0.107      0.506      0.314
FC              414     211      39    89.015      0.095      0.510      0.185
CU              314     127      29    80.024      0.072      0.404      0.228
ST              232      98      20    81.851      0.053      0.422      0.204
KC              136      60      22      82.2      0.031      0.441      0.367
FS              119      56      16     85.54      0.027      0.471      0.286
SV               25       9       0    79.056      0.006      0.360      0.000
NaN              16       0       0      83.0      0

In [4]:
print(
    by_pitch[by_pitch["swings"] >= 30]
    .sort_values("whiff_pct", ascending=False)
    [["pitches", "swings", "whiff_pct", "avg_velo"]]
    .round(3)
    .to_string()
)
print()
print("excluded (fewer than 30 swings):")
print(by_pitch[by_pitch["swings"] < 30][["pitches", "swings", "whiffs"]].to_string())

            pitches  swings  whiff_pct  avg_velo
pitch_type                                      
KC              136      60      0.367      82.2
SL              596     307      0.322    85.732
CH              466     236      0.314    86.198
FS              119      56      0.286     85.54
CU              314     127      0.228    80.024
ST              232      98      0.204    81.851
FC              414     211      0.185    89.015
FF             1346     626      0.165    94.218
SI              698     314      0.108      92.9

excluded (fewer than 30 swings):
            pitches  swings  whiffs
pitch_type                         
SV               25       9       0
NaN              16       0       0


In [5]:
unknown = df[df["pitch_type"].isna()]
print(unknown["description"].value_counts())
print()
print(unknown[["release_speed", "pfx_x", "pfx_z", "plate_x", "plate_z"]].isna().mean())

description
automatic_ball    16
Name: count, dtype: int64

release_speed    0.9375
pfx_x            0.9375
pfx_z            0.9375
plate_x          1.0000
plate_z          1.0000
dtype: float64


In [6]:
odd = df[df["pitch_type"].isna() & df["release_speed"].notna()]
print(odd[["game_pk", "inning", "at_bat_number", "pitch_number", "balls", "strikes",
           "release_speed", "pfx_x", "pfx_z", "plate_x", "plate_z", "des"]].to_string())

      game_pk  inning  at_bat_number  pitch_number  balls  strikes  release_speed  pfx_x  pfx_z  plate_x  plate_z  des
3618   746974       7             47             3      1        1           83.0  -1.54  -0.28     <NA>     <NA>  NaN


In [7]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))

from src.features.plate_discipline import whiff_rate, swing_rate, swinging_strike_rate
from src.data.validation import sort_chronologically, plate_appearances, check_first_pitch_counts

df = sort_chronologically(df)
check_first_pitch_counts(df)

print(f"Swing%: {swing_rate(df):.1%}")
print(f"Whiff%: {whiff_rate(df):.1%}")
print(f"SwStr%: {swinging_strike_rate(df):.1%}")

pa = plate_appearances(df)
print(f"PAs: {len(pa)}, dropped incomplete: {pa.attrs['incomplete_pas_dropped']}")

Swing%: 46.9%
Whiff%: 21.3%
SwStr%: 10.0%
PAs: 1110, dropped incomplete: 1
